In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.gait.loader import load_fit

acts = pd.read_csv("../data/raw/activities.csv")
print(acts.shape)
acts.columns.tolist()

(1177, 103)


['Activity ID',
 'Activity Date',
 'Activity Name',
 'Activity Type',
 'Activity Description',
 'Elapsed Time',
 'Distance',
 'Max Heart Rate',
 'Relative Effort',
 'Commute',
 'Activity Private Note',
 'Activity Gear',
 'Filename',
 'Athlete Weight',
 'Bike Weight',
 'Elapsed Time.1',
 'Moving Time',
 'Distance.1',
 'Max Speed',
 'Average Speed',
 'Elevation Gain',
 'Elevation Loss',
 'Elevation Low',
 'Elevation High',
 'Max Grade',
 'Average Grade',
 'Average Positive Grade',
 'Average Negative Grade',
 'Max Cadence',
 'Average Cadence',
 'Max Heart Rate.1',
 'Average Heart Rate',
 'Max Watts',
 'Average Watts',
 'Calories',
 'Max Temperature',
 'Average Temperature',
 'Relative Effort.1',
 'Total Work',
 'Number of Runs',
 'Uphill Time',
 'Downhill Time',
 'Other Time',
 'Perceived Exertion',
 'Type',
 'Start Time',
 'Weighted Average Power',
 'Power Count',
 'Prefer Perceived Exertion',
 'Perceived Relative Effort',
 'Commute.1',
 'Total Weight Lifted',
 'From Upload',
 'Grade Adj

In [2]:
acts["Activity Type"].value_counts()

Activity Type
Weight Training      482
Run                  448
Ride                 132
Swim                  72
Workout               34
Hike                   3
Rock Climb             2
Walk                   2
Football (Soccer)      2
Name: count, dtype: int64

In [3]:
runs = acts[acts["Activity Type"] == "Run"].copy()
runs = runs[runs["Filename"].notna()]
runs["date"] = pd.to_datetime(runs["Activity Date"], format="mixed")
runs["year"] = runs["date"].dt.year

print(len(runs))
runs.groupby("year").size()

446


year
2020      1
2021      5
2022     14
2023     56
2024    133
2025    149
2026     88
dtype: int64

In [4]:
long_runs = runs[runs["Moving Time"] > 600].copy()
print(len(long_runs))
long_runs.groupby("year").size()

441


year
2021      4
2022     12
2023     56
2024    132
2025    149
2026     88
dtype: int64

In [5]:
from src.gait.batch import parse_activities

sample = long_runs.groupby("year").head(4)
paths = ["../data/raw/" + name for name in sample["Filename"]]
print(len(paths))

frames, summary, failures = parse_activities(paths)
print(len(frames), "parsed,", len(failures), "failed")
failures

24
19 parsed, 5 failed


[{'path': '../data/raw/activities/8255072341.gpx',
  'error': "FitHeaderError('not a FIT file @ 0')"},
 {'path': '../data/raw/activities/8173972080.gpx',
  'error': "FitHeaderError('not a FIT file @ 0')"},
 {'path': '../data/raw/activities/8153468262.gpx',
  'error': "FitHeaderError('not a FIT file @ 0')"},
 {'path': '../data/raw/activities/8087356423.gpx',
  'error': "FitHeaderError('not a FIT file @ 0')"},
 {'path': '../data/raw/activities/5515941977.gpx.gz',
  'error': "FitHeaderError('not a FIT file @ 0')"}]

In [6]:
long_runs["is_fit"] = long_runs["Filename"].str.contains(".fit")
long_runs.groupby(["year", "is_fit"]).size()

year  is_fit
2021  False       1
      True        3
2022  False      12
2023  False      19
      True       37
2024  False       1
      True      131
2025  True      149
2026  True       88
dtype: int64

In [7]:
fit_runs = long_runs[long_runs["is_fit"]].copy()
sample = fit_runs.groupby("year").head(4)
paths = ["../data/raw/" + name for name in sample["Filename"]]

frames, summary, failures = parse_activities(paths)
print(len(frames), "parsed,", len(failures), "failed")
summary

19 parsed, 0 failed


,activity_id,n_samples,start_time,wall_clock_s,median_gap_s,max_gap_s,n_gaps_over_30s,paused_s,distance_m,has_speed,has_cadence,has_altitude
0,21307189990,6418,2026-09-13 09:35:41+00:00,6417.0,1.0,1.0,0,0.0,20891.91,True,True,True
1,21294934710,1217,2026-09-12 06:16:07+00:00,1216.0,1.0,1.0,0,0.0,3229.25,True,True,True
2,21260601745,1209,2026-09-10 09:10:02+00:00,1208.0,1.0,1.0,0,0.0,3212.45,True,True,True
3,21234799841,1437,2026-09-08 15:02:38+00:00,1436.0,1.0,1.0,0,0.0,4016.20,True,True,True
4,17958788319,1819,2025-12-28 16:12:16+00:00,1818.0,1.0,1.0,0,0.0,6128.76,True,True,True
5,17926961316,4276,2025-12-25 09:03:33+00:00,4275.0,1.0,1.0,0,0.0,12678.82,True,True,True
6,17908504806,2745,2025-12-23 06:44:56+00:00,2744.0,1.0,1.0,0,0.0,8082.94,True,True,True
7,17849366411,2479,2025-12-16 14:53:55+00:00,2478.0,1.0,1.0,0,0.0,7034.37,True,True,True
8,14112367314,1366,2024-12-30 07:15:07+00:00,1365.0,1.0,1.0,0,0.0,3633.26,True,True,True
9,14026852440,3578,2024-12-20 06:27:54+00:00,4256.0,1.0,528.0,2,681.0,10109.07,True,True,True


In [8]:
with open("../data/raw/activities/8255072341.gpx") as f:
    text = f.read(3000)
print(text)

<?xml version="1.0" encoding="UTF-8"?>
<gpx creator="StravaGPX" version="1.1" xmlns="http://www.topografix.com/GPX/1/1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.topografix.com/GPX/1/1 http://www.topografix.com/GPX/1/1/gpx.xsd">
 <metadata>
  <time>2022-12-16T13:26:04Z</time>
 </metadata>
 <trk>
  <name>Post Exam Run</name>
  <type>running</type>
  <trkseg>
   <trkpt lat="35.3182300" lon="33.3439820">
    <ele>86.6</ele>
    <time>2022-12-16T13:26:04Z</time>
   </trkpt>
   <trkpt lat="35.3182340" lon="33.3439940">
    <ele>86.6</ele>
    <time>2022-12-16T13:26:05Z</time>
   </trkpt>
   <trkpt lat="35.3182370" lon="33.3440040">
    <ele>86.6</ele>
    <time>2022-12-16T13:26:06Z</time>
   </trkpt>
   <trkpt lat="35.3182400" lon="33.3440140">
    <ele>86.6</ele>
    <time>2022-12-16T13:26:07Z</time>
   </trkpt>
   <trkpt lat="35.3182430" lon="33.3440240">
    <ele>86.6</ele>
    <time>2022-12-16T13:26:08Z</time>
   </trkpt>
   <trkpt lat="35.3182

In [9]:
for name in ["8255072341.gpx", "8173972080.gpx", "8087356423.gpx"]:
    with open("../data/raw/activities/" + name) as f:
        text = f.read().lower()
    print(name, "cad" in text, "hr" in text)

8255072341.gpx False False
8173972080.gpx False True
8087356423.gpx False True


In [10]:
import time

all_paths = ["../data/raw/" + name for name in fit_runs["Filename"]]
print(len(all_paths), "files to parse")

start = time.time()
frames, summary, failures = parse_activities(all_paths)
print(f"{len(frames)} parsed, {len(failures)} failed in {time.time() - start:.0f} s")

408 files to parse
408 parsed, 0 failed in 1006 s


In [11]:
tidy = pd.concat(frames, ignore_index=True)
print(tidy.shape)

tidy.to_parquet("../data/processed/tidy_samples.parquet")
summary.to_parquet("../data/processed/activity_summary.parquet")

(1325123, 9)


In [12]:
tidy.to_pickle("../data/processed/tidy_samples.pkl")
summary.to_pickle("../data/processed/activity_summary.pkl")
print("saved")

saved


In [13]:
import pandas as pd
print(pd.__version__)
import pyarrow
print(pyarrow.__version__)

tidy = pd.read_pickle("../data/processed/tidy_samples.pkl")
summary = pd.read_pickle("../data/processed/activity_summary.pkl")

tidy.to_parquet("../data/processed/tidy_samples.parquet")
summary.to_parquet("../data/processed/activity_summary.parquet")
print(tidy.shape, "written")

3.0.6
25.0.1
(1325123, 9) written


In [14]:
summary["year"] = summary["start_time"].dt.year

qc = summary.groupby("year").agg(
    activities=("activity_id", "count"),
    samples=("n_samples", "sum"),
    median_gap_s=("median_gap_s", "median"),
    pct_1hz=("median_gap_s", lambda s: (s == 1).mean() * 100),
    with_cadence=("has_cadence", "sum"),
    with_speed=("has_speed", "sum"),
    with_altitude=("has_altitude", "sum"),
    runs_with_pauses=("n_gaps_over_30s", lambda s: (s > 0).sum()),
    paused_hours=("paused_s", lambda s: s.sum() / 3600),
    distance_km=("distance_m", lambda s: s.sum() / 1000),
).round(1)

qc

,activities,samples,median_gap_s,pct_1hz,with_cadence,with_speed,with_altitude,runs_with_pauses,paused_hours,distance_km
year,,,,,,,,,,
2021,3,598,6.0,0.0,3,3,3,1,0.1,9.0
2023,37,97761,1.0,83.8,37,37,29,7,0.4,279.2
2024,131,422483,1.0,100.0,131,131,121,33,2.7,1193.4
2025,149,511942,1.0,100.0,149,149,145,17,1.4,1449.7
2026,88,292339,1.0,100.0,88,88,78,10,0.5,856.8


In [15]:
print(summary[~summary["has_speed"]][["activity_id", "start_time", "n_samples"]])
print(summary[~summary["has_cadence"]][["activity_id", "start_time", "n_samples"]])

Empty DataFrame
Columns: [activity_id, start_time, n_samples]
Index: []
Empty DataFrame
Columns: [activity_id, start_time, n_samples]
Index: []
